# AIC 2026 — Full Corpus ASR Colab-only v1

Notebook này dùng để chạy Whisper `medium` cho **toàn bộ corpus video**, không còn giới hạn ở 95 candidate.

Thiết kế:

- Đọc video từ folder share `AIC_2026`, không ghi gì vào đó.
- Quét toàn bộ `data_extracted/video` để tạo manifest full corpus.
- Mặc định chia thành 12 batch Colab, khoảng 70–75 video/batch nếu corpus khoảng 873 video.
- Có thể **skip/reuse các video đã transcribe thành công ở pilot batch 0** bằng `REUSE_PILOT_DONE = True`.
- Theo dõi tiến độ theo video: `done/total`, `%`, OK/lỗi, video hiện tại/kế tiếp.

Lưu ý: repo runner hiện chỉ cập nhật progress sau mỗi video hoàn tất, nên không có % nội bộ trong lúc một video đang được transcribe.


In [ ]:
# ===== 1) User config =====
REPO_URL = "https://github.com/doraIaIa/aic2026.git"
REPO_DIR = "/content/aic2026"

# Folder share/read-only chứa data gốc.
SHARED_DATA_ROOT = "/content/drive/MyDrive/AI_challenge/AIC_2026"

# Folder riêng của bạn để lưu shard/output/artifact.
MY_WORK_ROOT = "/content/drive/MyDrive/AI_challenge"
ARTIFACT_ROOT = f"{MY_WORK_ROOT}/artifacts"

VIDEO_SUBDIR = "data_extracted/video"
VIDEO_EXTENSIONS = [".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v"]

# Full corpus settings.
FULL_BATCH_COUNT = 12
BATCH_INDEX = 0  # chạy lần lượt 0..11
EXPECTED_CORPUS_VIDEOS = 873

# Skip các video đã chạy OK trong pilot batch trước đó.
REUSE_PILOT_DONE = True
PILOT_ARTIFACT_ROOT = f"{ARTIFACT_ROOT}/asr/whisper-medium-vi-pilot-v1-colab-only"

FULL_MANIFEST_DIR = f"{MY_WORK_ROOT}/asr/manifests/full-corpus-v1"
FULL_SHARD_DIR = f"{MY_WORK_ROOT}/asr/shards/full-corpus-v1-colab-{FULL_BATCH_COUNT:03d}"
OUTPUT_ROOT = f"{ARTIFACT_ROOT}/asr/whisper-medium-vi-full-v1-colab"
MERGED_OUTPUT_ROOT = f"{ARTIFACT_ROOT}/asr/whisper-medium-vi-full-v1-colab-merged"
CONFIG_PATH = "/content/aic-runtime.toml"

MODEL = "medium"
MODEL_REVISION = "medium"
LANGUAGE = "vi"
BEAM_SIZE = 5
VAD_FILTER = True
DEVICE = "auto"

# Smoke cho batch full hiện tại trước nếu muốn. Nếu đã tin pipeline, đặt RUN_FULL_BATCH=True.
SMOKE_LIMIT = 3
RUN_FULL_BATCH = True

print("SHARED_DATA_ROOT:", SHARED_DATA_ROOT)
print("MY_WORK_ROOT:", MY_WORK_ROOT)
print("FULL_BATCH_COUNT:", FULL_BATCH_COUNT)
print("BATCH_INDEX:", BATCH_INDEX)
print("REUSE_PILOT_DONE:", REUSE_PILOT_DONE)
print("RUN_FULL_BATCH:", RUN_FULL_BATCH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


In [ ]:
# ===== 2) Mount Google Drive =====
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
for p in [MY_WORK_ROOT, ARTIFACT_ROOT, FULL_MANIFEST_DIR, FULL_SHARD_DIR, OUTPUT_ROOT, MERGED_OUTPUT_ROOT]:
    Path(p).mkdir(parents=True, exist_ok=True)
print("Drive mounted and dirs ready.")


In [ ]:
# ===== 3) Clone/update repo and install dependencies =====
from pathlib import Path
import subprocess, sys, shutil

repo = Path(REPO_DIR)
if repo.exists():
    print("Repo exists:", REPO_DIR)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-8"], check=False)

subprocess.run(["apt-get", "update", "-y"], check=False)
subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "faster-whisper"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[asr]"], check=True)

print("ffmpeg:", shutil.which("ffmpeg"))
subprocess.run(["nvidia-smi"], check=False)
subprocess.run([sys.executable, "-m", "aic2026.cli", "run-asr-shard", "--help"], check=False)


In [ ]:
# ===== 4) Write correct runtime config =====
from pathlib import Path

for name, value in {
    "SHARED_DATA_ROOT": SHARED_DATA_ROOT,
    "MY_WORK_ROOT": MY_WORK_ROOT,
}.items():
    p = Path(value)
    print(name, value, "exists =", p.exists())
    if not p.exists():
        raise FileNotFoundError(f"{name} không tồn tại: {value}")

video_root = Path(SHARED_DATA_ROOT) / VIDEO_SUBDIR
print("VIDEO_ROOT:", video_root, "exists =", video_root.exists())
if not video_root.exists():
    raise FileNotFoundError("Không thấy thư mục video: " + str(video_root))

config_lines = [
    "[paths]",
    f"data_root = {SHARED_DATA_ROOT!r}",
    f"work_root = {MY_WORK_ROOT!r}",
    f"artifact_root = {ARTIFACT_ROOT!r}",
    "",
]
Path(CONFIG_PATH).write_text(chr(10).join(config_lines), encoding="utf-8")
print(Path(CONFIG_PATH).read_text(encoding="utf-8"))


In [ ]:
# ===== 5) Scan full corpus videos =====
from pathlib import Path
import json, hashlib

video_root = Path(SHARED_DATA_ROOT) / VIDEO_SUBDIR
data_root = Path(SHARED_DATA_ROOT)
video_paths = []
for ext in VIDEO_EXTENSIONS:
    video_paths.extend(video_root.rglob("*" + ext))
video_paths = sorted(set(p for p in video_paths if p.is_file()))

print("Scanned video files:", len(video_paths))
if EXPECTED_CORPUS_VIDEOS and len(video_paths) != EXPECTED_CORPUS_VIDEOS:
    print("WARNING: expected", EXPECTED_CORPUS_VIDEOS, "but found", len(video_paths))

items = []
seen = {}
for p in video_paths:
    video_id = p.stem
    if video_id in seen:
        raise ValueError(f"Trùng video_id {video_id}: {seen[video_id]} và {p}")
    seen[video_id] = str(p)
    items.append({
        "schema_version": 1,
        "video_id": video_id,
        "video_path": p.relative_to(data_root).as_posix(),
        "source": "full_corpus_scan_v1",
    })

print("First 5 video items:")
for item in items[:5]:
    print(item)

full_manifest_path = Path(FULL_MANIFEST_DIR) / "full-corpus-v1.jsonl"
full_manifest_path.write_text("".join(json.dumps(item, ensure_ascii=False, sort_keys=True) + chr(10) for item in items), encoding="utf-8")
FULL_MANIFEST_PATH = str(full_manifest_path)
FULL_CORPUS_TOTAL = len(items)
FULL_MANIFEST_SHA256 = hashlib.sha256(full_manifest_path.read_bytes()).hexdigest()
print("FULL_MANIFEST_PATH:", FULL_MANIFEST_PATH)
print("FULL_CORPUS_TOTAL:", FULL_CORPUS_TOTAL)
print("FULL_MANIFEST_SHA256:", FULL_MANIFEST_SHA256)


In [ ]:
# ===== 6) Find already completed pilot videos to skip/reuse =====
from pathlib import Path
import json

completed_pilot_ids = set()
pilot_root = Path(PILOT_ARTIFACT_ROOT)
print("PILOT_ARTIFACT_ROOT:", pilot_root, "exists =", pilot_root.exists())
if REUSE_PILOT_DONE and pilot_root.exists():
    for done_path in sorted(pilot_root.glob("*/DONE.json")):
        artifact_dir = done_path.parent
        try:
            marker = json.loads(done_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        videos_path = artifact_dir / "asr_videos.jsonl"
        if marker.get("task") == "asr_whisper" and videos_path.exists():
            with videos_path.open("r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        row = json.loads(line)
                        if row.get("status") == "OK" and row.get("video_id"):
                            completed_pilot_ids.add(row["video_id"])
            print("Reusable pilot artifact:", artifact_dir.name, "videos so far =", len(completed_pilot_ids))

print("Completed/reusable pilot videos:", len(completed_pilot_ids))
run_items = [item for item in items if item["video_id"] not in completed_pilot_ids]
print("Full corpus total:", FULL_CORPUS_TOTAL)
print("Videos to run now:", len(run_items))
print("Videos reused/skipped:", FULL_CORPUS_TOTAL - len(run_items))

RUN_ITEMS = run_items
RUN_TOTAL = len(run_items)


In [ ]:
# ===== 7) Split remaining videos into full-corpus Colab batches =====
from pathlib import Path
import json, hashlib

if not (0 <= BATCH_INDEX < FULL_BATCH_COUNT):
    raise ValueError("BATCH_INDEX phải nằm trong 0..FULL_BATCH_COUNT-1")
if RUN_TOTAL == 0:
    raise ValueError("Không còn video nào cần chạy")
if FULL_BATCH_COUNT > RUN_TOTAL:
    raise ValueError("FULL_BATCH_COUNT lớn hơn số video cần chạy")

base_count, rem = divmod(RUN_TOTAL, FULL_BATCH_COUNT)
created = []
start = 0
Path(FULL_SHARD_DIR).mkdir(parents=True, exist_ok=True)
config_hash = hashlib.sha256(json.dumps({"batch_count": FULL_BATCH_COUNT, "reuse_pilot_done": REUSE_PILOT_DONE}, sort_keys=True).encode("utf-8")).hexdigest()
for idx in range(FULL_BATCH_COUNT):
    count = base_count + (1 if idx < rem else 0)
    batch_items = RUN_ITEMS[start:start+count]
    start += count
    shard_id = f"full_batch_{idx:03d}_of_{FULL_BATCH_COUNT:03d}"
    payload = {
        "schema_version": 1,
        "task": "asr_whisper",
        "shard_id": shard_id,
        "shard_index": idx,
        "shard_count": FULL_BATCH_COUNT,
        "source_manifest": Path(FULL_MANIFEST_PATH).name,
        "source_manifest_sha256": FULL_MANIFEST_SHA256,
        "config_hash": config_hash,
        "git_commit": "notebook-generated",
        "git_dirty": True,
        "item_count": len(batch_items),
        "items": batch_items,
    }
    out = Path(FULL_SHARD_DIR) / f"{shard_id}.json"
    out.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    created.append((out, len(batch_items)))

print("FULL CORPUS RUN PLAN")
print("corpus total:", FULL_CORPUS_TOTAL)
print("reused/skipped:", FULL_CORPUS_TOTAL - RUN_TOTAL)
print("remaining to run:", RUN_TOTAL)
for idx, (path, count) in enumerate(created):
    marker = "<-- selected" if idx == BATCH_INDEX else ""
    print(f"batch {idx}: {count}/{RUN_TOTAL} remaining videos | {path.name} {marker}")

SELECTED_SHARD_PATH = str(created[BATCH_INDEX][0])
SELECTED_SHARD_NAME = created[BATCH_INDEX][0].stem
SELECTED_BATCH_TOTAL = created[BATCH_INDEX][1]
print("SELECTED_SHARD_PATH:", SELECTED_SHARD_PATH)
print("SELECTED_BATCH_TOTAL:", SELECTED_BATCH_TOTAL, "remaining videos")


In [ ]:
# ===== 8) Preflight selected batch video paths =====
from pathlib import Path
import json

raw = json.loads(Path(SELECTED_SHARD_PATH).read_text(encoding="utf-8"))
selected_items = raw["items"]
data_root = Path(SHARED_DATA_ROOT)
missing = []
for item in selected_items:
    p = data_root / item["video_path"]
    if not p.exists():
        missing.append((item["video_id"], p))

print(f"Selected batch: {SELECTED_BATCH_TOTAL}/{RUN_TOTAL} remaining videos")
print(f"Exists: {SELECTED_BATCH_TOTAL - len(missing)}/{SELECTED_BATCH_TOTAL}")
if missing:
    print("First 10 missing:")
    for vid, p in missing[:10]:
        print(vid, p)
    raise FileNotFoundError("Có video path không tồn tại")
print("Preflight OK")


In [ ]:
# ===== 9) Inspect existing selected batch artifact =====
from pathlib import Path
import json

if RUN_FULL_BATCH:
    OUT_DIR = Path(OUTPUT_ROOT) / SELECTED_SHARD_NAME
    limit_args = []
    EFFECTIVE_TOTAL = SELECTED_BATCH_TOTAL
else:
    OUT_DIR = Path(OUTPUT_ROOT) / f"smoke_{SELECTED_SHARD_NAME}_limit{SMOKE_LIMIT}"
    limit_args = ["--limit-videos", str(SMOKE_LIMIT)]
    EFFECTIVE_TOTAL = min(SMOKE_LIMIT, SELECTED_BATCH_TOTAL)

print("OUT_DIR:", OUT_DIR)
print("This run covers:", EFFECTIVE_TOTAL, "videos in selected batch")
print("Against full corpus:", FULL_CORPUS_TOTAL - RUN_TOTAL, "reused +", EFFECTIVE_TOTAL, "running now /", FULL_CORPUS_TOTAL)
print("DONE exists:", (OUT_DIR / "DONE.json").exists())
print("progress exists:", (OUT_DIR / "progress.json").exists())
if (OUT_DIR / "progress.json").exists():
    progress = json.loads((OUT_DIR / "progress.json").read_text(encoding="utf-8"))
    done = len(progress.get("completed_video_ids", []))
    failed = len(progress.get("failed_video_ids", []))
    print(f"Existing progress: {done} OK + {failed} failed / {EFFECTIVE_TOTAL}")


In [ ]:
# ===== 10) Run selected full-corpus batch with progress =====
from pathlib import Path
import subprocess, sys, time, json

OUT_DIR.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, "-m", "aic2026.cli", "run-asr-shard",
    "--config", CONFIG_PATH,
    "--shard", SELECTED_SHARD_PATH,
    "--out-dir", str(OUT_DIR),
    "--model", MODEL,
    "--model-revision", MODEL_REVISION,
    "--language", LANGUAGE,
    "--beam-size", str(BEAM_SIZE),
    "--device", DEVICE,
]
if VAD_FILTER:
    cmd.append("--vad-filter")
cmd.extend(limit_args)

print("Running:")
print(" ".join(cmd))
print(f"Batch progress target: {EFFECTIVE_TOTAL} videos")
print(f"Full-corpus accounting: {FULL_CORPUS_TOTAL - RUN_TOTAL} already reused/skipped, {RUN_TOTAL} remaining split over {FULL_BATCH_COUNT} batches")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
start_time = time.time()
last_line = None
all_ids = [item["video_id"] for item in json.loads(Path(SELECTED_SHARD_PATH).read_text(encoding="utf-8"))["items"][:EFFECTIVE_TOTAL]]

while proc.poll() is None:
    done_ids = []
    failed_ids = []
    progress_path = OUT_DIR / "progress.json"
    if progress_path.exists():
        try:
            progress = json.loads(progress_path.read_text(encoding="utf-8"))
            done_ids = progress.get("completed_video_ids", [])
            failed_ids = progress.get("failed_video_ids", [])
        except Exception:
            pass
    finished = len(done_ids) + len(failed_ids)
    pct = 100 * finished / max(EFFECTIVE_TOTAL, 1)
    finished_set = set(done_ids) | set(failed_ids)
    pending = [vid for vid in all_ids if vid not in finished_set]
    current = pending[0] if pending else "finalizing"
    elapsed_min = (time.time() - start_time) / 60
    global_done_est = (FULL_CORPUS_TOTAL - RUN_TOTAL) + finished
    global_pct = 100 * global_done_est / max(FULL_CORPUS_TOTAL, 1)
    line = f"batch {finished}/{EFFECTIVE_TOTAL} ({pct:.1f}%) | OK={len(done_ids)} failed={len(failed_ids)} | next/current={current} | full approx {global_done_est}/{FULL_CORPUS_TOTAL} ({global_pct:.1f}%) | elapsed={elapsed_min:.1f}m"
    if line != last_line:
        print(line, flush=True)
        last_line = line
    time.sleep(20)

stdout, stderr = proc.communicate()
print("RETURN CODE:", proc.returncode)
print("===== STDOUT last 8000 chars =====")
print(stdout[-8000:])
print("===== STDERR last 8000 chars =====")
print(stderr[-8000:])
if proc.returncode != 0:
    raise RuntimeError("run-asr-shard failed; xem STDERR phía trên")
print("Output dir:", OUT_DIR)


In [ ]:
# ===== 11) Validate and preview selected batch =====
from pathlib import Path
import subprocess, sys, json

subprocess.run([sys.executable, "-m", "aic2026.cli", "validate-asr-shard", "--artifact-dir", str(OUT_DIR)], check=True)
marker = json.loads((OUT_DIR / "DONE.json").read_text(encoding="utf-8"))
print("DONE summary:")
for key in ["shard_id", "expected_videos", "processed_videos", "failed_videos", "segment_count", "status", "device", "compute_type"]:
    print(key, "=", marker.get(key))

print("First 10 transcript segments:")
seg = OUT_DIR / "asr_segments.jsonl"
if seg.exists():
    with seg.open("r", encoding="utf-8") as f:
        for i, line in zip(range(10), f):
            row = json.loads(line)
            print(f"[{row.get('start_sec')} -> {row.get('end_sec')}] {row.get('video_id')}: {row.get('text')}")


In [ ]:
# ===== 12) Zip selected batch artifact =====
from pathlib import Path
import zipfile

zip_path = OUT_DIR.with_suffix(".zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in OUT_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(OUT_DIR.parent))
print("ZIP artifact:", zip_path)


In [ ]:
# ===== 13) Merge completed full batches + reused pilot artifacts =====
from pathlib import Path
import json, hashlib, datetime

merge_dir = Path(MERGED_OUTPUT_ROOT)
merge_dir.mkdir(parents=True, exist_ok=True)

artifact_dirs = []
# Reuse pilot completed artifacts first.
if REUSE_PILOT_DONE and Path(PILOT_ARTIFACT_ROOT).exists():
    artifact_dirs.extend(sorted(p.parent for p in Path(PILOT_ARTIFACT_ROOT).glob("*/DONE.json")))
# Add full-corpus completed batches.
artifact_dirs.extend(sorted(p.parent for p in Path(OUTPUT_ROOT).glob("*/DONE.json")))

print("artifact dirs considered:", len(artifact_dirs))
video_rows_by_id = {}
segments_by_video = {}
errors = []
used_artifacts = []
for artifact in artifact_dirs:
    try:
        marker = json.loads((artifact / "DONE.json").read_text(encoding="utf-8"))
    except Exception:
        continue
    if marker.get("task") != "asr_whisper":
        continue
    used_artifacts.append(str(artifact))
    videos_path = artifact / "asr_videos.jsonl"
    segments_path = artifact / "asr_segments.jsonl"
    errors_path = artifact / "errors.jsonl"
    if videos_path.exists():
        with videos_path.open("r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    row = json.loads(line)
                    vid = row.get("video_id")
                    if vid and vid not in video_rows_by_id:
                        video_rows_by_id[vid] = row
    if segments_path.exists():
        with segments_path.open("r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    row = json.loads(line)
                    vid = row.get("video_id")
                    if vid and vid not in segments_by_video:
                        segments_by_video.setdefault(vid, []).append(row)
                    elif vid in segments_by_video:
                        segments_by_video[vid].append(row)
    if errors_path.exists() and errors_path.stat().st_size:
        with errors_path.open("r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    errors.append(json.loads(line))

videos = [video_rows_by_id[k] for k in sorted(video_rows_by_id)]
segments = []
for vid in sorted(segments_by_video):
    segments.extend(segments_by_video[vid])

for name, rows in [("asr_videos.jsonl", videos), ("asr_segments.jsonl", segments), ("errors.jsonl", errors)]:
    (merge_dir / name).write_text("".join(json.dumps(r, ensure_ascii=False, sort_keys=True) + chr(10) for r in rows), encoding="utf-8")

checksums = {}
for name in ["asr_videos.jsonl", "asr_segments.jsonl", "errors.jsonl"]:
    checksums[name] = hashlib.sha256((merge_dir / name).read_bytes()).hexdigest()
summary = {
    "schema_version": 1,
    "task": "asr_whisper_full_merge",
    "created_at": datetime.datetime.utcnow().isoformat() + "Z",
    "corpus_total": FULL_CORPUS_TOTAL,
    "merged_video_count": len(videos),
    "merged_segment_count": len(segments),
    "error_count": len(errors),
    "used_artifacts": used_artifacts,
    "output_checksums": checksums,
}
(merge_dir / "DONE.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2)[:4000])
print("MERGED_OUTPUT_ROOT:", merge_dir)


## How to run full corpus on Colab

1. Nếu đang muốn bỏ qua 24 video pilot đã xong, giữ `REUSE_PILOT_DONE = True`.
2. Chạy batch full đầu tiên với `BATCH_INDEX = 0`.
3. Sau khi xong, đổi `BATCH_INDEX = 1`, rồi chạy lại từ cell 7 trở xuống.
4. Lặp đến batch cuối: mặc định `BATCH_INDEX = 11`.
5. Sau vài batch hoặc cuối cùng, chạy cell merge để gom full batches + reused pilot artifacts.

Nếu Colab hay ngắt, giữ `FULL_BATCH_COUNT = 12`. Nếu muốn batch nhỏ hơn nữa, tăng lên `16` hoặc `24` trước khi tạo shard.
